# 04 — Retrieval lexical, dense et fusion RRF

**Objectif** : observer les scores, les rangs et l'intérêt des requêtes par champ.

**Critère de passage** : les preuves requises apparaissent dans les candidats du champ correspondant ; la fusion ne masque pas les scores sources.

In [1]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()
from app.domain.profiles import map_question
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.planner import plan_queries
from app.store.artifacts import store

In [2]:
question = 'Quels indicateurs publics décrivent la position du Groupe en 2025 ?'
document_ids = ['foyer_financial_information_2025']
profile = map_question(question)
queries = plan_queries(question, profile, document_ids)
display_table([
    {'champ': query.field_id, 'requête': query.text}
    for query in queries
])

,champ,requête
0,group_equity,Quels indicateurs publics décrivent la positio...
1,non_life_market_share,Quels indicateurs publics décrivent la positio...
2,business_areas,Quels indicateurs publics décrivent la positio...


In [3]:
chunks = store.selected_chunks(document_ids)
retriever = HybridRetriever(chunks)
rows = []
for query in queries:
    for candidate in retriever.search(query, k=3):
        rows.append({
            'champ': query.field_id,
            'chunk': candidate.chunk.id,
            'rang_lexical': candidate.score.lexical_rank,
            'rang_dense': candidate.score.dense_rank,
            'score_rrf': candidate.score.rrf_score,
            'fait_attendu': any(fact.field_id == query.field_id for fact in candidate.chunk.facts),
        })
display_table(rows)

,champ,chunk,rang_lexical,rang_dense,score_rrf,fait_attendu
0,group_equity,fin-2025-equity,1,1,0.032787,True
1,group_equity,fin-2025-market,2,2,0.032258,False
2,group_equity,fin-2025-activities,3,3,0.031746,False
3,non_life_market_share,fin-2025-market,1,1,0.032787,True
4,non_life_market_share,fin-2025-activities,2,2,0.032258,False
5,non_life_market_share,fin-2025-equity,3,3,0.031746,False
6,business_areas,fin-2025-activities,1,1,0.032787,True
7,business_areas,fin-2025-equity,2,3,0.032002,False
8,business_areas,fin-2025-market,3,2,0.032002,False


In [4]:
recall_rows = []
for k in (1, 2, 3, 5):
    recovered = 0
    for query in queries:
        candidates = retriever.search(query, k=k)
        recovered += int(any(
            fact.field_id == query.field_id
            for candidate in candidates
            for fact in candidate.chunk.facts
        ))
    recall_rows.append({'k': k, 'recall_by_required_field': recovered / len(queries)})
assert recall_rows[-1]['recall_by_required_field'] == 1.0
display_table(recall_rows)

,k,recall_by_required_field
0,1,1.0
1,2,1.0
2,3,1.0
3,5,1.0


### Point de contrôle

Une question unique peut exiger plusieurs preuves. Mesurer `recall@k` par champ obligatoire est plus utile ici qu'un score global de similarité sur la question entière.